## Human Value Detection

### Importar Librerías

In [1]:
#!pip install requests tqdm scikit-learn

In [1]:
import ssl
import os
import re
import json
import time
import ollama
import requests
import pandas as pd
from dataclasses import dataclass
from tqdm.auto import tqdm
from typing import Any, Dict, List, Optional, Tuple
from sklearn.metrics import f1_score, accuracy_score
ssl._create_default_https_context = ssl._create_unverified_context

### Espacios de etiquetas (19 subcategorías + 4 categorías + NONE)

In [ ]:
# 19 subcategorías (tu multilabel)
SUB19 = [
    "Self-Direction: Thought",
    "Self-Direction: Action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power: Dominance",
    "Power: Resources",
    "Face",
    "Security: Personal",
    "Security: Societal",
    "Tradition",
    "Conformity: Rules",
    "Conformity: Interpersonal",
    "Humility",
    "Benevolence: Caring",
    "Benevolence: Dependability",
    "Universalism: Concern",
    "Universalism: Nature",
    "Universalism: Tolerance",
]

# 4 categorías (para multiclass)
CAT4 = [
    "Openness to Change",
    "Self-Enhancement",
    "Conservation",
    "Self-Transcendence",
]

# mapping subcategoría -> categoría (ajústalo si tu criterio difiere)
SUB_TO_CAT = {}
for sub in ["Self-Direction: Thought","Self-Direction: Action","Stimulation","Hedonism"]:
    SUB_TO_CAT[sub] = "Openness to Change"
for sub in ["Achievement","Power: Dominance","Power: Resources","Face"]:
    SUB_TO_CAT[sub] = "Self-Enhancement"
for sub in ["Security: Personal","Security: Societal","Tradition",
            "Conformity: Rules","Conformity: Interpersonal","Humility"]:
    SUB_TO_CAT[sub] = "Conservation"
for sub in ["Benevolence: Caring","Benevolence: Dependability",
            "Universalism: Concern","Universalism: Nature","Universalism: Tolerance"]:
    SUB_TO_CAT[sub] = "Self-Transcendence"

MC5 = CAT4 + ["NONE"]


### Cliente Ollama (HTTP) + extractor JSON robusto

In [3]:
@dataclass
class OllamaConfig:
    base_url: str = "http://localhost:11434"
    timeout_s: int = 120
    temperature: float = 0.0
    top_p: float = 1.0
    num_ctx: int = 4096

class OllamaClient:
    def __init__(self, cfg: OllamaConfig):
        self.cfg = cfg

    def generate(self, model: str, prompt: str) -> str:
        url = f"{self.cfg.base_url}/api/generate"
        payload = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": self.cfg.temperature,
                "top_p": self.cfg.top_p,
                "num_ctx": self.cfg.num_ctx,
            },
        }
        r = requests.post(url, json=payload, timeout=self.cfg.timeout_s)
        r.raise_for_status()
        return r.json().get("response", "")

_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)

def extract_json(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    m = _JSON_BLOCK_RE.search(text)
    if not m:
        raise ValueError(f"No se encontró bloque JSON. Respuesta: {text[:200]}")
    raw = m.group(0).strip()
    repaired = raw
    repaired = re.sub(r"(?<!\\)'", '"', repaired)      # comillas simples -> dobles
    repaired = re.sub(r",\s*([}\]])", r"\1", repaired) # trailing commas
    return json.loads(repaired)

def as_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, str):
        return [x]
    if isinstance(x, list):
        return [str(v) for v in x]
    return [str(x)]


### Tipos de prompt (multilabel vs. multiclass)

In [4]:
SYSTEM = (
    "Eres un clasificador de valores humanos. "
    "Devuelve SOLO un JSON válido, sin texto extra, sin markdown."
)

PROMPTS = {
    # 1) Zero-shot mínimo
    "zero_shot": """{sys}
Tarea: Clasifica la frase según el esquema indicado.

Esquema: {schema_desc}

Etiquetas permitidas (usa exactamente estos nombres):
{label_list}

Devuelve SOLO este JSON:
{json_schema}

Frase: "{sentence}"
""",

    # 2) Con pistas (definiciones cortas)
    "with_defs": """{sys}
Tarea: Clasifica la frase.

Pistas rápidas:
- Openness to Change: autonomía, novedad, placer
- Self-Enhancement: logro, poder, estatus, reputación
- Conservation: seguridad, tradición, conformidad, modestia
- Self-Transcendence: benevolencia, igualdad, tolerancia, naturaleza

Esquema: {schema_desc}

Etiquetas permitidas:
{label_list}

Devuelve SOLO este JSON:
{json_schema}

Frase: "{sentence}"
""",

    # 3) Few-shot
    "few_shot": """{sys}
Tarea: Clasifica la frase (esquema indicado).

Esquema: {schema_desc}
Etiquetas permitidas:
{label_list}

Formato salida:
{json_schema}

Ejemplos:
Frase: "Me encanta explorar sitios nuevos sin plan."
Salida: {{"values":["Stimulation","Self-Direction: Action"]}}

Frase: "Las normas están para cumplirse; si no, esto es un caos."
Salida: {{"values":["Conformity: Rules","Security: Societal"]}}

Frase: "Hay que ayudar a los demás aunque cueste."
Salida: {{"values":["Benevolence: Caring"]}}

Ahora:
Frase: "{sentence}"
""",

    # 4) Estricto
    "strict": """{sys}
Reglas:
- SOLO puedes usar etiquetas de la lista cerrada.
- Si no hay evidencia clara, devuelve ausencia (lista vacía o "NONE" según esquema).
- No inventes etiquetas.

Esquema: {schema_desc}
Etiquetas:
{label_list}

Devuelve SOLO este JSON:
{json_schema}

Frase: "{sentence}"
""",
}

def build_prompt(prompt_type: str, sentence: str, task: str) -> str:
    """
    task: "multilabel_sub19" o "multiclass_cat5"
    """
    if task == "multilabel_sub19":
        label_list = "\n".join([f"- {x}" for x in SUB19])
        schema_desc = "Multilabel (0..N etiquetas) sobre 19 subcategorías."
        json_schema = '{ "values": ["<subvalue1>", "<subvalue2>", ...] }'
    elif task == "multiclass_cat5":
        label_list = "\n".join([f"- {x}" for x in MC5])
        schema_desc = "Multiclass (1 etiqueta) sobre 4 categorías + NONE."
        json_schema = '{ "class": "<one_of_5_classes>" }'
    else:
        raise ValueError(f"task desconocida: {task}")

    template = PROMPTS[prompt_type]
    return template.format(
        sys=SYSTEM,
        schema_desc=schema_desc,
        label_list=label_list,
        json_schema=json_schema,
        sentence=str(sentence).replace('"', '\\"'),
    )


### Carga dataset + detección columnas (texto + gold)

In [5]:
def load_tsv(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", dtype=str).fillna("")

def detect_text_column(df: pd.DataFrame) -> str:
    for c in ["sentence", "text", "content", "tweet", "utterance"]:
        if c in df.columns:
            return c
    return df.columns[0]

def parse_gold_list(cell: str) -> List[str]:
    cell = (cell or "").strip()
    if not cell:
        return []
    if cell.startswith("[") and cell.endswith("]"):
        try:
            arr = json.loads(cell)
            return [str(x) for x in arr]
        except Exception:
            pass
    parts = re.split(r"\s*[,;|]\s*", cell)
    return [p.strip() for p in parts if p.strip()]

def detect_gold_column(df: pd.DataFrame) -> Optional[str]:
    for c in ["labels", "values", "human_values", "gold", "label"]:
        if c in df.columns:
            return c
    return None

def gold_sub19_to_mc5(gold_subs: List[str]) -> str:
    """
    Convierte gold multilabel sub19 -> 1 clase (cat4 o NONE).
    Regla práctica: si hay varias categorías activas, elige la más frecuente; empate -> primera por orden.
    Ajusta si tu definición de multiclass es distinta.
    """
    if not gold_subs:
        return "NONE"
    cats = [SUB_TO_CAT.get(s) for s in gold_subs if s in SUB_TO_CAT]
    cats = [c for c in cats if c is not None]
    if not cats:
        return "NONE"
    # mayoría
    counts = {c: cats.count(c) for c in set(cats)}
    best = sorted(counts.items(), key=lambda x: (-x[1], CAT4.index(x[0]) if x[0] in CAT4 else 999))[0][0]
    return best


### Métricas (multilabel y multiclass)

In [6]:
def multilabel_to_multihot(labels: List[str], space: List[str]) -> List[int]:
    s = set(labels)
    return [1 if lab in s else 0 for lab in space]

def metrics_multilabel_sub19(y_true: List[List[str]], y_pred: List[List[str]]) -> Dict[str, float]:
    Yt = [multilabel_to_multihot(v, SUB19) for v in y_true]
    Yp = [multilabel_to_multihot(v, SUB19) for v in y_pred]

    # micro-F1 sobre todos los bits
    micro = f1_score(sum(Yt, []), sum(Yp, []), average="binary", zero_division=0)

    # macro-F1 por etiqueta
    f1s = []
    for j in range(len(SUB19)):
        tj = [row[j] for row in Yt]
        pj = [row[j] for row in Yp]
        f1s.append(f1_score(tj, pj, average="binary", zero_division=0))
    macro = float(sum(f1s) / len(f1s))

    return {"micro_f1": float(micro), "macro_f1": float(macro)}

def metrics_multiclass_mc5(y_true: List[str], y_pred: List[str]) -> Dict[str, float]:
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro", labels=MC5, zero_division=0)
    return {"accuracy": float(acc), "macro_f1": float(macro)}


### Runner general: eliges modelo, prompt_type y experimento

In [7]:
def run_experiment(
    dataset_path: str,
    model: str,
    prompt_type: str,
    experiment: str,
    max_rows: Optional[int] = None,
    sleep_s: float = 0.0,
    ollama_url: str = "http://localhost:11434",
    save_jsonl_path: Optional[str] = None,
) -> Dict[str, Any]:
    """
    experiment:
      - "multilabel_sub19"  -> predice {"values":[...]} y calcula micro/macro F1 si hay gold
      - "multiclass_mc5"    -> predice {"class":"..."} y calcula acc/macro-F1 si hay gold (derivado)
    """
    df = load_tsv(dataset_path)
    text_col = detect_text_column(df)
    gold_col = detect_gold_column(df)

    if max_rows is not None:
        df = df.head(max_rows)

    gold_sub19 = None
    if gold_col is not None:
        gold_sub19 = [parse_gold_list(x) for x in df[gold_col].tolist()]

    client = OllamaClient(OllamaConfig(base_url=ollama_url))

    preds_sub19: List[List[str]] = []
    preds_mc5: List[str] = []
    failures = 0

    jsonl_f = open(save_jsonl_path, "w", encoding="utf-8") if save_jsonl_path else None

    try:
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Inferencia"):
            sent = str(row[text_col])

            # Construcción de prompt según experimento
            if experiment == "multilabel_sub19":
                task = "multilabel_sub19"
            elif experiment == "multiclass_mc5":
                task = "multiclass_cat5"
            else:
                raise ValueError(f"Experimento no soportado: {experiment}")

            prompt = build_prompt(prompt_type, sent, task=task)

            last_err = None
            parsed = None
            raw = None

            for attempt in range(3):
                try:
                    raw = client.generate(model=model, prompt=prompt)
                    parsed = extract_json(raw)
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(0.4 * (attempt + 1))

            if parsed is None:
                failures += 1
                if experiment == "multilabel_sub19":
                    preds_sub19.append([])
                    pred_obj = {"values": []}
                else:
                    preds_mc5.append("NONE")
                    pred_obj = {"class": "NONE"}

                rec = {
                    "id": int(idx),
                    "text": sent,
                    "pred": pred_obj,
                    "error": str(last_err),
                    "model": model,
                    "prompt_type": prompt_type,
                    "experiment": experiment,
                }
            else:
                if experiment == "multilabel_sub19":
                    vals = [v for v in as_list(parsed.get("values")) if v in SUB19]
                    preds_sub19.append(vals)
                    pred_obj = {"values": vals}
                else:
                    c = str(parsed.get("class", "NONE"))
                    if c not in MC5:
                        c = "NONE"
                    preds_mc5.append(c)
                    pred_obj = {"class": c}

                rec = {
                    "id": int(idx),
                    "text": sent,
                    "pred": pred_obj,
                    "raw": raw,
                    "model": model,
                    "prompt_type": prompt_type,
                    "experiment": experiment,
                }

            # añadir gold si existe
            if gold_sub19 is not None:
                rec["gold_sub19"] = gold_sub19[len(preds_sub19)-1] if experiment == "multilabel_sub19" else gold_sub19[len(preds_mc5)-1]

            if jsonl_f:
                jsonl_f.write(json.dumps(rec, ensure_ascii=False) + "\n")

            if sleep_s > 0:
                time.sleep(sleep_s)

    finally:
        if jsonl_f:
            jsonl_f.close()

    summary: Dict[str, Any] = {
        "dataset_path": dataset_path,
        "rows": len(df),
        "text_col": text_col,
        "gold_col": gold_col,
        "model": model,
        "prompt_type": prompt_type,
        "experiment": experiment,
        "failures": failures,
        "metrics": None,
    }

    # Métricas (si hay gold)
    if gold_sub19 is not None:
        if experiment == "multilabel_sub19":
            summary["metrics"] = metrics_multilabel_sub19(gold_sub19, preds_sub19)
        else:
            gold_mc5 = [gold_sub19_to_mc5(v) for v in gold_sub19]
            summary["metrics"] = metrics_multiclass_mc5(gold_mc5, preds_mc5)

    return summary

### Ejecución (selección modelo, prompt y experimento)

In [ ]:
DATASET = "sentences_train.tsv"   # ajusta ruta
MODEL = "llama3.1"                # ejemplo: "mistral", "qwen2.5", etc.
PROMPT_TYPE = "strict"            # zero_shot | with_defs | few_shot | strict

# 1) Multilabel (SUB19) con métricas micro/macro F1
summary_ml = run_experiment(
    dataset_path=DATASET,
    model=MODEL,
    prompt_type=PROMPT_TYPE,
    experiment="multilabel_sub19",
    max_rows=200,                 # None para todo
    save_jsonl_path="preds_multilabel.jsonl",
)
summary_ml


In [ ]:
# 2) Multiclass (MC5 = CAT4 + NONE) con accuracy y macro-F1
summary_mc = run_experiment(
    dataset_path=DATASET,
    model=MODEL,
    prompt_type=PROMPT_TYPE,
    experiment="multiclass_mc5",
    max_rows=200,
    save_jsonl_path="preds_multiclass.jsonl",
)
summary_mc


In [8]:
print('Hola, María')

Hola, María
